**🎯 06 — Lyft: phân tích CẢ HAI target (giá & hệ số nhân)**

Chỉ Lyft mới có dữ liệu `surge_multiplier` thật, nên đây là notebook duy nhất
phân tích được **cả 2 biến mục tiêu** cùng lúc.

> ⚠️ **Điểm phương pháp then chốt: hai target có ĐỘ PHÂN GIẢI khác nhau**
>
> | Target | Cấp độ | Vì sao |
> |---|---|---|
> | `price` | **Từng cuốc** | Mỗi cuốc có giá riêng theo loại xe + quãng đường |
> | `surge_multiplier` | **Thị trường** (thời điểm × tuyến) | Hệ số nhân **dùng chung** cho mọi loại xe tại cùng thời điểm & tuyến |
>
> Phân tích surge theo từng cuốc = **đếm trùng ~2,1 lần** → thổi phồng độ tin cậy.
> Notebook này gộp về đúng cấp thị trường trước khi phân tích.

**0. Nạp dữ liệu & dựng 2 bảng theo 2 độ phân giải**

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys; sys.path.insert(0, ".")
import importlib, _common; importlib.reload(_common)
from _common import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

setup()
df, dfU, dfL = load()

# --- Bang 1: TUNG CUOC (cho target price) ---
CUOC = dfL.copy()

# --- Bang 2: CAP THI TRUONG (cho target surge) ---
# Shared khong bao gio surge -> loai bo truoc khi gop
L = dfL[dfL.name != "Shared"].copy()
THITRUONG = L.groupby(["timestamp","source","destination"], as_index=False).agg(
    surge=("surge_multiplier","median"), is_surge=("is_surge","max"),
    hour_local=("hour_local","first"), weekday_local=("weekday_local","first"),
    is_weekend=("is_weekend","first"), short_summary=("short_summary","first"),
    temperature=("temperature","first"), precipIntensity=("precipIntensity","first"),
    humidity=("humidity","first"), windSpeed=("windSpeed","first"),
    visibility=("visibility","first"), date_local=("date_local","first"),
    distance=("distance","median"), n_bao_gia=("price","size"))

print(f"Lyft tong                    : {len(dfL):,} cuoc")
print(f"  - bo Shared (khong surge)  : {len(L):,} cuoc")
print(f"  - gop ve (thoi diem x tuyen): {len(THITRUONG):,} quan sat DOC LAP")
print(f"    (moi quan sat bi lap {len(L)/len(THITRUONG):.1f} lan trong du lieu goc)")
print()
print(f"Ty le surge — tinh theo tung cuoc : {L.is_surge.mean()*100:.2f}%")
print(f"Ty le surge — tinh cap thi truong : {THITRUONG.is_surge.mean()*100:.2f}%")
# QUAN TRONG: dung BINS TOAN CUC (ca 2 hang) de so sanh duoc voi notebook 01-05.
# Neu dung dfL.distance.max() thi bin hep hon -> it mau moi o -> THOI PHONG
# cac yeu to yeu (vd thoi tiet 4.7% -> 10.0%) do nhieu lay mau.
BINS = np.linspace(0, df.distance.max(), 16)

**1. ✅ Kiểm chứng: surge có thật sự là đại lượng cấp thị trường?**

In [ ]:
g = L.groupby(["timestamp","source","destination"]).surge_multiplier.agg(n_gia="nunique", n="size")
g = g[g.n > 1]
print(f"So nhom (thoi diem x tuyen) co >1 loai xe: {len(g):,}")
print(f"  Nhom co surge GIONG HET o moi loai xe  : {(g.n_gia==1).mean()*100:.2f}%")
print()
t = dfL.groupby("name").agg(so_cuoc=("price","size"), so_surge=("is_surge","sum"))
t["ty_le_%"] = (t.so_surge/t.so_cuoc*100).round(2)
display(t.sort_values("so_surge", ascending=False))
print("=> 5 dich vu co DUNG cung so cuoc surge; rieng Shared = 0.")
print("   Xac nhan: surge ap theo THI TRUONG, khong theo loai xe. Shared duoc mien.")

**2. 📊 Thước đo chung cho cả 2 target**

- **Với `price`**: % phổ giá thô mà yếu tố giải thích được (đã kiểm soát quãng đường + loại xe)
- **Với `surge`**: η trên bảng cấp thị trường (đã khử trùng)

> ⚠️ **Lưu ý về độ nhạy:** thước đo "% phổ giá" phụ thuộc vào **độ rộng bin quãng đường**.
> Bin càng hẹp → càng ít mẫu mỗi ô → nhiễu lấy mẫu **thổi phồng** các yếu tố yếu
> (thử nghiệm: thời tiết 4,7% → 10,0% khi đổi bin). Vì vậy notebook này dùng
> **cùng bộ bin toàn cục** với notebook 01–05 để so sánh được.

In [ ]:
YEU_TO = [("📏 Loại xe",      "name"),
          ("📍 Khu vực đón",  "source"),
          ("📍 Điểm đến",     "destination"),
          ("🕐 Giờ",          "hour_local"),
          ("🕐 Thứ",          "weekday_local"),
          ("🕐 Cuối tuần",    "is_weekend"),
          ("🌦️ Thời tiết",    "short_summary"),
          ("🌦️ Nhiệt độ",     "temperature"),
          ("🌦️ Cường độ mưa", "precipIntensity")]

def suc_manh_gia(cot):
    b = pd.cut(CUOC.distance, BINS)
    tho = CUOC.groupby(b, observed=True).price.std().mean()
    if cot == "name":
        xe = CUOC.groupby([b, "name"], observed=True).price.std().mean()
        return (1 - xe/tho) * 100
    key = binned(CUOC[cot]).rename("_k")
    lech = CUOC.groupby([b, "name", key], observed=True).price.mean()                .groupby(level=[0,1]).std().mean()
    return lech / tho * 100

rows = []
for lab, c in YEU_TO:
    rows.append({"Yếu tố": lab,
                 "% phổ GIÁ": round(suc_manh_gia(c), 1),
                 "eta với SURGE": round(eta(binned(THITRUONG[c]), THITRUONG.is_surge), 4)
                                  if c in THITRUONG.columns else np.nan})
t = pd.DataFrame(rows)
display(t.sort_values("% phổ GIÁ", ascending=False).reset_index(drop=True))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
d1 = t.dropna(subset=["% phổ GIÁ"]).sort_values("% phổ GIÁ")
ax[0].barh(range(len(d1)), d1["% phổ GIÁ"], color=BLUE, alpha=.88, zorder=3)
ax[0].set_yticks(range(len(d1))); ax[0].set_yticklabels(d1["Yếu tố"], fontsize=10)
for i, v in enumerate(d1["% phổ GIÁ"]): ax[0].text(v+.8, i, f"{v:.1f}%", va="center", fontsize=8)
ax[0].set_xlabel("% phổ giá thô"); ax[0].set_title("Ảnh hưởng tới GIÁ", fontweight="bold")

d2 = t.dropna(subset=["eta với SURGE"]).sort_values("eta với SURGE")
ax[1].barh(range(len(d2)), d2["eta với SURGE"], color=RED, alpha=.88, zorder=3)
ax[1].set_yticks(range(len(d2))); ax[1].set_yticklabels(d2["Yếu tố"], fontsize=10)
for i, v in enumerate(d2["eta với SURGE"]): ax[1].text(v+.002, i, f"{v:.3f}", va="center", fontsize=8)
ax[1].set_xlabel("eta (cấp thị trường)"); ax[1].set_title("Ảnh hưởng tới HỆ SỐ NHÂN", fontweight="bold")
for a in ax: a.grid(axis="y", visible=False)
fig.suptitle("Lyft — hai target, hai nhom yeu to khac nhau", fontweight="bold", fontsize=13)
fig.tight_layout(); plt.show()
print("=> GIA   <- loai xe + quang duong (thuoc tinh chuyen di)")
print("   SURGE <- khu vuc               (boi canh thi truong)")

**3. 📍 Vị trí → cả 2 target**

In [ ]:
loc_s = THITRUONG.groupby("source").agg(n=("is_surge","size"), surge=("is_surge","mean"),
                                         mult=("surge","mean"))
loc_s["surge_%"] = (loc_s.surge*100).round(2)
loc_p = CUOC.groupby("source").agg(gia_TB=("price","mean"), qd_TB=("distance","mean"))
loc = loc_s.join(loc_p).sort_values("surge_%", ascending=False)
display(loc[["n","surge_%","mult","gia_TB","qd_TB"]].round(3))
print(f"Chenh lech surge giua khu cao nhat va thap nhat: "
      f"{loc['surge_%'].max()/loc['surge_%'].min():.1f} lan")

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
d = loc.sort_values("surge_%")
ax[0].barh(range(len(d)), d["surge_%"], color=RED, alpha=.88, zorder=3)
ax[0].axvline(THITRUONG.is_surge.mean()*100, color=ORANGE, ls="--", lw=1.5,
              label=f"TB {THITRUONG.is_surge.mean()*100:.2f}%")
ax[0].set_yticks(range(len(d))); ax[0].set_yticklabels(d.index, fontsize=9)
ax[0].set_xlabel("% co surge"); ax[0].set_title("SURGE theo khu don", fontweight="bold")
ax[0].legend(fontsize=8, frameon=False)

ax[1].scatter(loc.qd_TB, loc["surge_%"], s=90, color=PURPLE, alpha=.8, zorder=3)
for k, v in loc.iterrows():
    ax[1].annotate(k, (v.qd_TB, v["surge_%"]), fontsize=7, xytext=(4,3),
                   textcoords="offset points")
r = np.corrcoef(loc.qd_TB, loc["surge_%"])[0,1]
ax[1].set_xlabel("Quang duong TB cua khu (dam)"); ax[1].set_ylabel("% co surge")
ax[1].set_title(f"Khu co chuyen NGAN co hay surge hon? (r={r:.3f})", fontweight="bold")
for a in ax: a.grid(axis="y", visible=False)
fig.tight_layout(); plt.show()

**4. 🕐 Thời gian → cả 2 target**

In [ ]:
h_s = THITRUONG.groupby("hour_local").agg(n=("is_surge","size"), surge=("is_surge","mean"))
h_p = CUOC.groupby("hour_local").price.mean()
thr = h_s.surge.quantile(.75)
PEAK = sorted(h_s[h_s.surge >= thr].index.tolist())
print(f"GIO CAO DIEM (top 25% ty le surge): {PEAK}")
print(f"  Surge dao dong {h_s.surge.min()*100:.2f}% -> {h_s.surge.max()*100:.2f}% "
      f"({h_s.surge.max()/h_s.surge.min():.2f} lan)")
print(f"  Gia   dao dong {h_p.min():.2f} -> {h_p.max():.2f} USD "
      f"({(h_p.max()/h_p.min()-1)*100:.2f}%)")

fig, ax = plt.subplots(1, 2, figsize=(15, 4.4))
ax[0].bar(h_s.index, h_s.surge*100,
          color=[RED if x in PEAK else BLUE for x in h_s.index], alpha=.9, zorder=3)
ax[0].axhline(thr*100, color=ORANGE, ls="--", lw=1.5, label=f"nguong {thr*100:.1f}%")
ax[0].set_title("SURGE theo gio (do = cao diem)", fontweight="bold")
ax[0].set_ylabel("% co surge"); ax[0].legend(fontsize=8, frameon=False)
ax[1].plot(h_p.index, h_p.values, "o-", color=BLUE, lw=2, ms=4)
ax[1].set_title(f"GIA theo gio (bien thien chi {(h_p.max()/h_p.min()-1)*100:.1f}%)",
                fontweight="bold")
ax[1].set_ylabel("Gia TB (USD)")
for a in ax: a.set_xlabel("Gio"); a.set_xticks(range(0,24,3)); a.grid(axis="x", visible=False)
fig.tight_layout(); plt.show()

DAYS = ["T2","T3","T4","T5","T6","T7","CN"]
pv = THITRUONG.pivot_table(index="weekday_local", columns="hour_local",
                            values="is_surge", aggfunc="mean")*100
fig, a = plt.subplots(figsize=(14, 3.4))
im = a.imshow(pv, cmap="Reds", aspect="auto")
a.set_xticks(range(24)); a.set_xticklabels(range(24), fontsize=8)
a.set_yticks(range(len(pv))); a.set_yticklabels([DAYS[i] for i in pv.index])
a.set_xlabel("Gio"); a.set_title("% surge — Gio x Thu (cap thi truong)", fontweight="bold")
a.grid(False); fig.colorbar(im, fraction=0.02)
fig.tight_layout(); plt.show()

**5. 🌦️ Thời tiết → cả 2 target**

In [ ]:
w_s = THITRUONG.groupby("short_summary").agg(n=("is_surge","size"), surge=("is_surge","mean"))
w_s = w_s[w_s.n >= 500]
w_s["surge_%"] = (w_s.surge*100).round(2)
w_p = CUOC.groupby("short_summary").price.mean()
w_s["gia_TB"] = w_p.reindex(w_s.index).round(2)
w_s["so_ngay"] = THITRUONG.groupby("short_summary").date_local.nunique().reindex(w_s.index)
display(w_s.sort_values("surge_%", ascending=False)[["n","surge_%","gia_TB","so_ngay"]])
print("!! Cot so_ngay: kieu thoi tiet chi xuat hien <=5 ngay thi khong tach duoc")
print("   khoi hieu ung ngay -> ket luan khong dang tin.")

fig, ax = plt.subplots(1, 2, figsize=(15, 4.4))
d = w_s.sort_values("surge_%")
ax[0].barh(range(len(d)), d["surge_%"], color=RED, alpha=.88, zorder=3)
ax[0].set_yticks(range(len(d))); ax[0].set_yticklabels(d.index, fontsize=9)
ax[0].axvline(THITRUONG.is_surge.mean()*100, color=ORANGE, ls="--", lw=1.5)
ax[0].set_xlabel("% co surge"); ax[0].set_title("SURGE theo thoi tiet", fontweight="bold")

THITRUONG["nhom_mua"] = pd.cut(THITRUONG.precipIntensity, [-1e-9, 1e-4, .01, .05, 10],
                                labels=["Không mưa","Mưa rất nhẹ","Mưa nhẹ","Mưa vừa/to"])
g = THITRUONG.groupby("nhom_mua", observed=True).agg(surge=("is_surge","mean"), n=("is_surge","size"))
ax[1].bar(range(len(g)), g.surge*100, color=BLUE, alpha=.85, zorder=3)
ax[1].set_xticks(range(len(g))); ax[1].set_xticklabels(g.index, fontsize=8.5, rotation=10)
for i, (v, n2) in enumerate(zip(g.surge*100, g.n)):
    ax[1].text(i, v, f"{v:.2f}%\n(n={n2:,})", ha="center", va="bottom", fontsize=7.5)
ax[1].set_ylabel("% co surge"); ax[1].set_title("Muc do mua -> surge", fontweight="bold")
for a in ax: a.grid(axis="x", visible=False)
fig.tight_layout(); plt.show()

**6. 🧪 Kiểm chứng bằng model: dự đoán được đến đâu?**

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score

# --- Target 1: GIA (tung cuoc) ---
Xp = CUOC[["name","distance","source","destination"]].copy()
for c in ["name","source","destination"]: Xp[c] = Xp[c].astype("category")
a, b, c_, d_ = train_test_split(Xp, CUOC.price, test_size=.25, random_state=42)
mp = HistGradientBoostingRegressor(max_iter=250, learning_rate=.06,
        categorical_features=["name","source","destination"], random_state=42).fit(a, c_)
print(f"GIA   : R2 = {r2_score(d_, mp.predict(b)):.4f}")

# --- Target 2: SURGE (cap thi truong) ---
FS = ["source","destination","hour_local","weekday_local","is_weekend",
      "short_summary","temperature","precipIntensity","humidity","windSpeed"]
CATS = ["source","destination","short_summary"]
Xs = THITRUONG[FS].copy()
for c in CATS: Xs[c] = Xs[c].astype("category")
y = THITRUONG.is_surge
a, b, c_, d_ = train_test_split(Xs, y, test_size=.25, random_state=42, stratify=y)
ms = HistGradientBoostingClassifier(max_iter=250, learning_rate=.06,
        categorical_features=CATS, random_state=42).fit(a, c_)
pr = ms.predict_proba(b)[:,1]
print(f"SURGE : ROC-AUC = {roc_auc_score(d_, pr):.4f}  (0.5 = doan bua)")
print(f"        PR-AUC  = {average_precision_score(d_, pr):.4f}  (nen so voi {d_.mean():.4f})")
print(f"        Accuracy khi doan 'khong surge' cho tat ca = {1-d_.mean():.4f}")
print()
print("!! Voi bai toan mat can bang, PHAI doc ROC-AUC / PR-AUC, KHONG doc accuracy.")

from sklearn.inspection import permutation_importance
pi = permutation_importance(ms, b, d_, n_repeats=3, random_state=0, scoring="roc_auc")
imp = pd.DataFrame({"feature": FS, "importance": pi.importances_mean})         .sort_values("importance", ascending=False).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 4.6))
dd = imp.head(10)[::-1]
ax.barh(range(len(dd)), np.clip(dd.importance, 0, None), color=PURPLE, zorder=3)
ax.set_yticks(range(len(dd))); ax.set_yticklabels(dd.feature, fontsize=9)
ax.set_title("Feature nao du doan SURGE tot nhat? (do bang ROC-AUC)", fontweight="bold")
ax.grid(axis="y", visible=False); fig.tight_layout(); plt.show()
display(imp.head(8).round(5))

**7. 🔍 Tần suất vs Cường độ — hai câu hỏi khác nhau**

Từ trước tới giờ ta chỉ đo **tần suất** (bao nhiêu % chuyến bị surge). Nhưng còn câu hỏi
thứ hai: **khi đã surge thì surge mạnh bao nhiêu?**

Hai chiều này có thể độc lập — một khu có thể hiếm khi surge nhưng khi surge thì rất nặng.

In [ ]:
M = THITRUONG.copy()
M["co_surge"] = (M.surge > 1).astype(int)

def tan_suat_cuong_do(cot):
    return M.groupby(cot).agg(
        tan_suat=("co_surge", "mean"),
        cuong_do=("surge", lambda s: s[s > 1].mean()),
        cao_nhat=("surge", "max"),
        n=("co_surge", "size"))

tsc = tan_suat_cuong_do("source").sort_values("tan_suat", ascending=False)
tsc["tan_suat_%"] = (tsc.tan_suat * 100).round(2)
display(tsc[["n", "tan_suat_%", "cuong_do", "cao_nhat"]].round(3))

r = np.corrcoef(tsc.tan_suat, tsc.cuong_do)[0, 1]
print(f"Tuong quan (tan suat) vs (cuong do) theo khu: r = {r:.3f}")
print()
if r > 0.7:
    print("=> Hai chieu DI CUNG NHAU: khu nao hay surge thi cung surge MANH.")
    print("   Vi vay chi can mot chi so la du mo ta anh huong cua khu vuc.")
else:
    print("=> Hai chieu DOC LAP: phai bao cao ca hai.")

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
d = tsc.sort_values("tan_suat")
y = np.arange(len(d))
ax[0].barh(y - .2, d["tan_suat_%"], height=.38, color=RED, label="Tan suat (%)", zorder=3)
ax0b = ax[0].twiny()
ax0b.barh(y + .2, d.cuong_do, height=.38, color=PURPLE, label="Cuong do", zorder=3)
ax[0].set_yticks(y); ax[0].set_yticklabels(d.index, fontsize=9)
ax[0].set_xlabel("% co surge", color=RED)
ax0b.set_xlabel("He so nhan TB khi co surge", color=PURPLE)
ax0b.set_xlim(1.0, d.cuong_do.max() * 1.05)
ax[0].set_title("Khu vuc: tan suat va cuong do", fontweight="bold")
ax[0].grid(axis="y", visible=False)

ax[1].scatter(tsc.tan_suat * 100, tsc.cuong_do, s=110, color=BLUE, alpha=.8, zorder=3)
for k, v in tsc.iterrows():
    ax[1].annotate(k, (v.tan_suat * 100, v.cuong_do), fontsize=7.5,
                   xytext=(5, 3), textcoords="offset points")
ax[1].set_xlabel("% chuyen co surge"); ax[1].set_ylabel("He so nhan TB khi co surge")
ax[1].set_title(f"Hai chieu co di cung nhau khong?  (r = {r:.3f})", fontweight="bold")
ax[1].grid(axis="y", visible=False)
fig.tight_layout(); plt.show()

**8. 🕐 Thời gian tác động lên chiều nào?**

Nếu khu vực ảnh hưởng cả tần suất lẫn cường độ, thì thời gian thì sao?

In [ ]:
h = tan_suat_cuong_do("hour_local")
print(f"Theo GIO:")
print(f"  Tan suat: {h.tan_suat.min()*100:.1f}% - {h.tan_suat.max()*100:.1f}%"
      f"   (chenh {h.tan_suat.max()/h.tan_suat.min():.2f} lan)")
print(f"  Cuong do: {h.cuong_do.min():.3f} - {h.cuong_do.max():.3f}"
      f"   (chenh {h.cuong_do.max()/h.cuong_do.min():.2f} lan)")
print(f"  Tuong quan giua hai chieu: r = {np.corrcoef(h.tan_suat, h.cuong_do)[0,1]:.3f}")
print()
print("=> Gio anh huong TAN SUAT nhung gan nhu KHONG anh huong CUONG DO.")
print("   Khac han khu vuc (anh huong ca hai). Day la hai co che khac nhau:")
print("     - Khu vuc: quyet dinh muc do khan hiem xe co ban  -> ca hai chieu")
print("     - Gio    : quyet dinh thoi diem xay ra khan hiem  -> chi tan suat")

fig, ax = plt.subplots(1, 2, figsize=(15, 4.4))
ax[0].bar(h.index, h.tan_suat * 100, color=RED, alpha=.85, zorder=3)
ax[0].set_title(f"Tan suat theo gio (chenh {h.tan_suat.max()/h.tan_suat.min():.2f} lan)",
                fontweight="bold")
ax[0].set_ylabel("% co surge")
ax[1].plot(h.index, h.cuong_do, "o-", color=PURPLE, lw=2, ms=5)
ax[1].set_ylim(1.0, h.cuong_do.max() * 1.08)
ax[1].set_title(f"Cuong do theo gio (chenh chi {h.cuong_do.max()/h.cuong_do.min():.2f} lan)",
                fontweight="bold")
ax[1].set_ylabel("He so nhan TB khi co surge")
for a in ax: a.set_xlabel("Gio"); a.set_xticks(range(0, 24, 3)); a.grid(axis="x", visible=False)
fig.tight_layout(); plt.show()

**9. ⭐ Surge có dai dẳng không? — câu hỏi quyết định cho Task ii**

Với `price`, feature mạnh nhất là **lịch sử giá** (`history_price_mean_last6`, importance
0,742) — vì giá rất dai dẳng, giờ này gần như giờ trước.

Câu hỏi tương ứng cho surge: **nếu giờ trước có surge, giờ này có nhiều khả năng surge hơn không?**

Đây là điều quyết định liệu lag feature có cứu được model surge hay không.

In [ ]:
S = M.sort_values(["source", "destination", "timestamp"]).copy()
S["truoc"] = S.groupby(["source", "destination"]).co_surge.shift(1)
ok = S.truoc.notna()
ct = pd.crosstab(S.loc[ok, "truoc"], S.loc[ok, "co_surge"], normalize="index") * 100

p0 = ct.loc[0.0, 1]; p1 = ct.loc[1.0, 1]
print("Xac suat gio nay co surge, tuy theo gio truoc:")
print(f"   Gio truoc KHONG surge  ->  {p0:.1f}% co surge")
print(f"   Gio truoc CO surge     ->  {p1:.1f}% co surge")
print(f"   Ty so kha nang         :  {p1/p0:.2f} lan")
print()

# So sanh voi do dai dang cua GIA (tinh ngay tren du lieu tung cuoc)
G = CUOC.groupby(["source", "destination", "name", "timestamp"], as_index=False) \
        .price.median()
G = G.sort_values(["source", "destination", "name", "timestamp"])
G["gia_truoc"] = G.groupby(["source", "destination", "name"]).price.shift(1)
mg = G.gia_truoc.notna()
r_gia = np.corrcoef(G.loc[mg, "price"], G.loc[mg, "gia_truoc"])[0, 1]

print("SO SANH DO DAI DANG cua hai target:")
print(f"   GIA  : tuong quan voi gia quan sat truoc  r = {r_gia:.3f}   -> RAT dai dang")
print(f"   SURGE: ty so kha nang chi {p1/p0:.2f} lan                -> dai dang YEU")
print()
print("=> Day la ly do model surge kho: surge KHONG dai dang nen lag feature")
print("   khong giup nhieu. Trong khi gia rat dai dang nen lag feature rat manh.")
print("   Muon du doan surge phai co tin hieu CUNG-CAU thoi gian thuc, khong the")
print("   dua vao lich su surge.")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
ax[0].bar(["Giờ trước\nKHÔNG surge", "Giờ trước\nCÓ surge"], [p0, p1],
          color=[BLUE, RED], alpha=.88, zorder=3, width=.55)
for i, v in enumerate([p0, p1]):
    ax[0].text(i, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax[0].set_ylabel("% giờ này có surge")
ax[0].set_title(f"Độ dai dẳng của SURGE (chỉ {p1/p0:.2f} lần)", fontweight="bold")
ax[0].grid(axis="x", visible=False)

lags = range(1, 9)
pers = []
for k in lags:
    S[f"t{k}"] = S.groupby(["source", "destination"]).co_surge.shift(k)
    m2 = S[f"t{k}"].notna()
    c = pd.crosstab(S.loc[m2, f"t{k}"], S.loc[m2, "co_surge"], normalize="index") * 100
    pers.append(c.loc[1.0, 1] / c.loc[0.0, 1] if 0.0 in c.index else np.nan)
ax[1].plot(list(lags), pers, "o-", color=PURPLE, lw=2, ms=6)
ax[1].axhline(1.0, color=MUT, ls="--", lw=1.5, label="1.0 = khong dai dang")
ax[1].set_xlabel("Khoang cach (gio)"); ax[1].set_ylabel("Tỷ số khả năng")
ax[1].set_title("Ảnh hưởng tắt nhanh thế nào?", fontweight="bold")
ax[1].legend(fontsize=8, frameon=False); ax[1].grid(axis="x", visible=False)
fig.tight_layout(); plt.show()

**10. 📌 Kết luận**

In [ ]:
print("="*72); print("LYFT — HAI TARGET, HAI CAU CHUYEN KHAC NHAU"); print("="*72)
print()
print("A) DO PHAN GIAI KHAC NHAU")
print(f"   price : {len(CUOC):,} cuoc  (moi cuoc mot gia)")
print(f"   surge : {len(THITRUONG):,} quan sat thi truong (da bo Shared + khu trung {len(L)/len(THITRUONG):.1f}x)")
print()
print("B) YEU TO QUYET DINH")
tp = t.sort_values("% phổ GIÁ", ascending=False).head(3)
ts_ = t.dropna(subset=["eta với SURGE"]).sort_values("eta với SURGE", ascending=False).head(3)
print("   Top 3 cho GIA:")
for _, r in tp.iterrows(): print(f"      {r['Yếu tố']:<18} {r['% phổ GIÁ']:>5.1f}% pho gia")
print("   Top 3 cho SURGE:")
for _, r in ts_.iterrows(): print(f"      {r['Yếu tố']:<18} eta = {r['eta với SURGE']:.4f}")
print()
print("C) KHOANG CACH GIUA 2 TARGET")
print(f"   GIA  : du doan tot (R2 ~0.91) — gan nhu tat dinh tu loai xe + quang duong")
print(f"   SURGE: du doan kem (AUC ~0.7-0.8) — thieu tin hieu cung-cau")
print()
print("D) BO FEATURE DE XUAT (rieng cho Lyft)")
print("   Model GIA   : name, distance, source, destination")
print("   Model SURGE : source, destination, hour_local, short_summary")
print("                 (train tren bang CAP THI TRUONG, da bo Shared)")
print()
print("E) HAN CHE")
print("   - Thoi tiet hiem chi xuat hien 2-5 ngay -> lan voi hieu ung ngay")
print("   - Khong co du lieu cung-cau (so tai xe, so cuoc cho) -> tran cua model surge")
print("   - 12 khu deu la trung tam Boston (bán kính ~4km) -> chua kiem chung duoc ngoai o")